# SA-DMAE Segmentation Fine-tuning

BraTS 2021 종양 segmentation (WT / TC / ET)  
두 가지 실험:
1. **DMAE encoder** (n_slices=1, axial_depth=0) + seg head
2. **SA-DMAE encoder** (n_slices=3, axial_depth=4) + seg head

> 런타임 → 런타임 유형 변경 → **GPU (T4)** 먼저 설정!

In [ ]:
# ── Cell 1: GPU 확인 & Drive 마운트 ──────────────────────────────────────────
import torch
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
    print('VRAM           :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 2: 코드 & 패키지 ────────────────────────────────────────────────────
import os

REPO_DIR = '/content/SA-DMAE'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/whkim4338/SA-DMAE.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!pip install timm nibabel tensorboard tqdm -q
print('완료')

In [ ]:
# ── Cell 3: 경로 설정 (여기만 수정) ─────────────────────────────────────────

# BraTS 2021 NIfTI 원본 경로 (Drive)
BRATS_NIFTI  = '/content/drive/MyDrive/BraTS2021'     # ← 실제 경로로 수정

# Pre-trained 체크포인트 경로
DMAE_CKPT    = '/content/drive/MyDrive/SA_DMAE_output/dmae_baseline/checkpoint-best.pth'
SA_DMAE_CKPT = '/content/drive/MyDrive/SA_DMAE_output/checkpoint-best.pth'

# 출력 디렉터리
OUTPUT_DMAE  = '/content/drive/MyDrive/SA_DMAE_seg/dmae'
OUTPUT_SA    = '/content/drive/MyDrive/SA_DMAE_seg/sa_dmae'

# 학습 설정
EPOCHS      = 50
BATCH_SIZE  = 8      # T4 기준 8 권장
LR          = 1e-4
VAL_RATIO   = 0.2
PATIENCE    = 15
SAVE_EVERY  = 10

import os
os.makedirs(OUTPUT_DMAE, exist_ok=True)
os.makedirs(OUTPUT_SA,   exist_ok=True)

# 데이터 확인
from pathlib import Path
cases = [p for p in Path(BRATS_NIFTI).iterdir() if p.is_dir()]
print(f'BraTS cases : {len(cases)}개')
print(f'DMAE ckpt   : {Path(DMAE_CKPT).exists()}')
print(f'SA-DMAE ckpt: {Path(SA_DMAE_CKPT).exists()}')

In [ ]:
# ── Cell 4: TensorBoard ───────────────────────────────────────────────────────
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/SA_DMAE_seg

In [ ]:
# ── Cell 5: 실험 1 — DMAE encoder (baseline) ──────────────────────────────────
!python main_finetune_seg.py \
    --data_path   {BRATS_NIFTI} \
    --resume      {DMAE_CKPT} \
    --n_slices    1 \
    --axial_depth 0 \
    --epochs      {EPOCHS} \
    --batch_size  {BATCH_SIZE} \
    --lr          {LR} \
    --val_ratio   {VAL_RATIO} \
    --patience    {PATIENCE} \
    --save_every  {SAVE_EVERY} \
    --output_dir  {OUTPUT_DMAE} \
    --log_dir     {OUTPUT_DMAE} \
    --device      cuda \
    --num_workers 2

In [ ]:
# ── Cell 5-R: 실험 1 이어서 학습 (세션 끊겼을 때) ────────────────────────────
import glob, os
ckpts = sorted(
    [f for f in glob.glob(f'{OUTPUT_DMAE}/checkpoint-*.pth') if 'best' not in f and 'final' not in f],
    key=os.path.getmtime
)
if ckpts:
    latest = ckpts[-1]
    print(f'이어서 학습: {latest}')
    !python main_finetune_seg.py \
        --data_path   {BRATS_NIFTI} \
        --resume      {DMAE_CKPT} \
        --resume_seg  {latest} \
        --n_slices    1 \
        --axial_depth 0 \
        --epochs      {EPOCHS} \
        --batch_size  {BATCH_SIZE} \
        --lr          {LR} \
        --val_ratio   {VAL_RATIO} \
        --patience    {PATIENCE} \
        --save_every  {SAVE_EVERY} \
        --output_dir  {OUTPUT_DMAE} \
        --log_dir     {OUTPUT_DMAE} \
        --device      cuda \
        --num_workers 2
else:
    print('저장된 체크포인트 없음. Cell 5를 먼저 실행하세요.')

In [ ]:
# ── Cell 6: 실험 2 — SA-DMAE encoder (ours) ───────────────────────────────────
!python main_finetune_seg.py \
    --data_path   {BRATS_NIFTI} \
    --resume      {SA_DMAE_CKPT} \
    --n_slices    3 \
    --axial_depth 4 \
    --epochs      {EPOCHS} \
    --batch_size  {BATCH_SIZE} \
    --lr          {LR} \
    --val_ratio   {VAL_RATIO} \
    --patience    {PATIENCE} \
    --save_every  {SAVE_EVERY} \
    --output_dir  {OUTPUT_SA} \
    --log_dir     {OUTPUT_SA} \
    --device      cuda \
    --num_workers 2

In [ ]:
# ── Cell 6-R: 실험 2 이어서 학습 (세션 끊겼을 때) ────────────────────────────
import glob, os
ckpts = sorted(
    [f for f in glob.glob(f'{OUTPUT_SA}/checkpoint-*.pth') if 'best' not in f and 'final' not in f],
    key=os.path.getmtime
)
if ckpts:
    latest = ckpts[-1]
    print(f'이어서 학습: {latest}')
    !python main_finetune_seg.py \
        --data_path   {BRATS_NIFTI} \
        --resume      {SA_DMAE_CKPT} \
        --resume_seg  {latest} \
        --n_slices    3 \
        --axial_depth 4 \
        --epochs      {EPOCHS} \
        --batch_size  {BATCH_SIZE} \
        --lr          {LR} \
        --val_ratio   {VAL_RATIO} \
        --patience    {PATIENCE} \
        --save_every  {SAVE_EVERY} \
        --output_dir  {OUTPUT_SA} \
        --log_dir     {OUTPUT_SA} \
        --device      cuda \
        --num_workers 2
else:
    print('저장된 체크포인트 없음. Cell 6을 먼저 실행하세요.')

In [ ]:
# ── Cell 7: 결과 비교 ─────────────────────────────────────────────────────────
import json
import matplotlib.pyplot as plt

def load_log(path):
    return [json.loads(l) for l in open(path)]

dmae_log = load_log(f'{OUTPUT_DMAE}/log_seg.txt')
sa_log   = load_log(f'{OUTPUT_SA}/log_seg.txt')

def best_row(logs):
    return max(logs, key=lambda d: d['dice_mean'])

dmae_best = best_row(dmae_log)
sa_best   = best_row(sa_log)

print('=' * 60)
print(f'{"":20s} {"WT":>8} {"TC":>8} {"ET":>8} {"Mean":>8}')
print('-' * 60)
print(f'{"DMAE":20s} {dmae_best["dice_wt"]:8.4f} {dmae_best["dice_tc"]:8.4f} {dmae_best["dice_et"]:8.4f} {dmae_best["dice_mean"]:8.4f}')
print(f'{"SA-DMAE":20s} {sa_best["dice_wt"]:8.4f} {sa_best["dice_tc"]:8.4f} {sa_best["dice_et"]:8.4f} {sa_best["dice_mean"]:8.4f}')
delta_wt   = sa_best['dice_wt']   - dmae_best['dice_wt']
delta_tc   = sa_best['dice_tc']   - dmae_best['dice_tc']
delta_et   = sa_best['dice_et']   - dmae_best['dice_et']
delta_mean = sa_best['dice_mean'] - dmae_best['dice_mean']
print('-' * 60)
print(f'{"SA-DMAE 개선":20s} {delta_wt:+8.4f} {delta_tc:+8.4f} {delta_et:+8.4f} {delta_mean:+8.4f}')
print('=' * 60)

# Loss & Dice curve
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for logs, label, color in [(dmae_log, 'DMAE', 'tab:blue'), (sa_log, 'SA-DMAE', 'tab:orange')]:
    epochs = [d['epoch'] + 1 for d in logs]
    axes[0].plot(epochs, [d['val_loss']  for d in logs], label=label, color=color)
    axes[1].plot(epochs, [d['dice_mean'] for d in logs], label=label, color=color)

axes[0].set_title('Val Loss');      axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_title('Val Dice Mean'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/SA_DMAE_seg/seg_comparison.png', dpi=120)
plt.show()

# Save metrics JSON
result = {
    'dmae':    {'dice_wt': dmae_best['dice_wt'], 'dice_tc': dmae_best['dice_tc'],
                'dice_et': dmae_best['dice_et'], 'dice_mean': dmae_best['dice_mean']},
    'sa_dmae': {'dice_wt': sa_best['dice_wt'],   'dice_tc': sa_best['dice_tc'],
                'dice_et': sa_best['dice_et'],   'dice_mean': sa_best['dice_mean']},
}
with open('/content/drive/MyDrive/SA_DMAE_seg/seg_metrics.json', 'w') as f:
    json.dump(result, f, indent=2)
print('저장 완료: seg_metrics.json')